<a href="https://colab.research.google.com/github/Adityasen-cmd/Autonomous_Vehicle_Lane_Detection/blob/main/Autonomous_Vehicle_Lane_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORTING PACKAGES

In [7]:
import cv2
import numpy as np
import math
!pip install ultralytics opencv-python-headless -q
from ultralytics import YOLO



 REGION OF INTEREST

In [8]:


def region_of_interest(img, vertices):
    mask = np.zeros_like(img)
    cv2.fillPoly(mask, vertices, 255)
    return cv2.bitwise_and(img, mask)



 DRAW LANE AREA

In [9]:


def draw_lane_lines(img, left_line, right_line,
                    color=(0,255,0), thickness=10):

    line_img = np.zeros_like(img)

    poly_pts = np.array([[
        (left_line[0], left_line[1]),
        (left_line[2], left_line[3]),
        (right_line[2], right_line[3]),
        (right_line[0], right_line[1])
    ]], dtype=np.int32)

    cv2.fillPoly(line_img, poly_pts, color)

    return cv2.addWeighted(img, 0.8, line_img, 0.5, 0)



 LANE DETECTION PIPELINE


In [10]:

def pipeline(image):

    height, width = image.shape[:2]

    roi_vertices = [
        (0, height),
        (width // 2, height // 2),
        (width, height)
    ]

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    canny = cv2.Canny(gray, 100, 200)

    cropped = region_of_interest(
        canny,
        np.array([roi_vertices], np.int32)
    )

    lines = cv2.HoughLinesP(
        cropped,
        rho=6,
        theta=np.pi/60,
        threshold=160,
        minLineLength=40,
        maxLineGap=25
    )

    if lines is None:
        return image

    left_x = []
    left_y = []
    right_x = []
    right_y = []

    for line in lines:
        for x1,y1,x2,y2 in line:

            if x2 == x1:
                continue

            slope = (y2-y1)/(x2-x1)

            if abs(slope) < 0.5:
                continue

            if slope < 0:
                left_x.extend([x1,x2])
                left_y.extend([y1,y2])
            else:
                right_x.extend([x1,x2])
                right_y.extend([y1,y2])

    min_y = int(height * 0.6)
    max_y = height

    if len(left_x) > 0:

        poly_left = np.poly1d(
            np.polyfit(left_y, left_x, 1)
        )

        left_start = int(poly_left(max_y))
        left_end = int(poly_left(min_y))

    else:
        left_start = left_end = 0

    if len(right_x) > 0:

        poly_right = np.poly1d(
            np.polyfit(right_y, right_x, 1)
        )

        right_start = int(poly_right(max_y))
        right_end = int(poly_right(min_y))

    else:
        right_start = right_end = 0

    lane_img = draw_lane_lines(
        image,
        [left_start, max_y, left_end, min_y],
        [right_start, max_y, right_end, min_y]
    )

    return lane_img



DISTANCE ESTIMATION

In [11]:

def estimate_distance(box_width):

    focal_length = 1000
    known_width = 2.0

    if box_width == 0:
        return 0

    return (known_width * focal_length) / box_width



 MAIN PROCESSING

In [12]:

def process_video():

    # Automatically downloads if absent
    model = YOLO("yolov8n.pt")

    input_video = "/content/video.mp4"
    output_video = "/content/output.mp4"

    cap = cv2.VideoCapture(input_video)

    if not cap.isOpened():
        print("Unable to open video.")
        return

    width = 1280
    height = 720
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 30

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    out = cv2.VideoWriter(
        output_video,
        fourcc,
        fps,
        (width, height)
    )

    frame_count = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.resize(frame, (width, height))

        lane_frame = pipeline(frame)

        results = model.predict(
            frame,
            verbose=False
        )

        for result in results:

            for box in result.boxes:

                cls = int(box.cls[0])
                conf = float(box.conf[0])

                if model.names[cls] == "car" and conf >= 0.5:

                    x1,y1,x2,y2 = map(
                        int,
                        box.xyxy[0]
                    )

                    cv2.rectangle(
                        lane_frame,
                        (x1,y1),
                        (x2,y2),
                        (0,255,255),
                        2
                    )

                    label = f"Car {conf:.2f}"

                    cv2.putText(
                        lane_frame,
                        label,
                        (x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0,255,255),
                        2
                    )

                    distance = estimate_distance(
                        x2-x1
                    )

                    cv2.putText(
                        lane_frame,
                        f"{distance:.1f} m",
                        (x1,y2+25),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (255,0,0),
                        2
                    )

        out.write(lane_frame)

        frame_count += 1

        if frame_count % 50 == 0:
            print(f"Processed {frame_count} frames")

    cap.release()
    out.release()


    print("Processing Complete!")
    print("Saved at:")
    print(output_video)

process_video()

Processed 50 frames
Processed 100 frames
Processed 150 frames
Processed 200 frames
Processed 250 frames
Processed 300 frames
Processed 350 frames
Processing Complete!
Saved at:
/content/output.mp4
